In [ ]:
from helical.models.geneformer import Geneformer, GeneformerConfig
import anndata as ad
import scanpy as sc
import random
import numpy as np

#load the model
model_config = GeneformerConfig(model_name="gf-12L-38M-i4096", batch_size=10)
geneformer = Geneformer(model_config)

#Load the data to test
ann_data = ad.read_h5ad("counts_combined_filtered_BA4_sALS_PN.h5ad")


# 1. In-Silico Perturbation Workflow

This script defines a function that can KO or KI a gene or multiple genes into existing cell data and generate the embeddings from the original cells and the modified cells for downstream comparison. A brief test KO of 2 highly abundant brain genes is also included.

Steps the function performs:
- Subset the given data based on cell metadata to the cells of interest for the perturbation (e.g. healthy or diseased, cell type etc.)
- Select a given number of cells at random that match that metadata for downstream modification
- Basic checks that all genes are seen in the subset cells and, in the case of KO, have some gene expression
- Generate a modified data object with the gene KO (expression set to 0) or KI (expression set to max of all genes within the cells)
- Return Geneformer embeddings for both the original and modified cells for downstream analysis

Further features that could be added:
- Combine KI and KO in the same experiment
- Handle automating gene combinations within the function (although a wrapper function might be more applicable)
- Mode to generate an "average cell" across cell populations rather than sub sampling random cells
- Option to handle combinations of sub grouping by metadata (donor, cell type etc., again could be solve with a wrapper or custom metadata)


In [205]:
#PERTURBATION FUNCTION
#1. Selects x number of cells from a given condition ie. health and disease
#2. Sets the expression of gene in cells to max or zero based on KI or KO
#3. Tokenise for Geneformer and return embeddings for the cells and their unmodified versions
    
def perturb(ann_dat, mod, cellfeature="", featuredef="", numcells=10, genes=["PDCD1"], direction="KO"):
    #id the cell types of interest
    targcells =  ann_data[ann_data.obs[cellfeature] == featuredef]
    #randomly subsample x cells
    subsamp = random.sample(range(len(targcells)),min(len(targcells),numcells))
    #subset adat
    sub_adat = targcells[subsamp]
    sc.pp.filter_genes(sub_adat, min_cells=1)
    #check if target gene is present and give error if not for KO
    if any(x not in sub_adat.var_names for x in genes) and direction == "KO":
        print("At least one gene not seen in any sampled cells.")
        return
    sums = sum(sub_adat.X[:, sub_adat.var_names.get_indexer(genes)])
    if sums.max()==0:
        print("Gene have zero expression in all sampled cells already.")
        return()
    #modify to ki or ko gene
    mod_adat = sub_adat.copy()
    if direction == "KO":
        mod_adat.X[:, mod_adat.var_names.get_indexer(genes)] = 0
    elif direction == "KI":
        mod_adat.X[:, mod_adat.var_names.get_indexer(genes)] = mod_adat.X.max()
    #generate the embeddings
    baseem = geneformer.get_embeddings(geneformer.process_data(sub_adat))
    modem = geneformer.get_embeddings(geneformer.process_data(mod_adat))
    embeds = {"base":baseem,"modified":modem,"params":[cellfeature,featuredef,numcells,genes,direction]}
    return(embeds)


## Test the function

A brief test using high expressed brain genes and just looking if the head of the embedding matrix looks altered by the gene changes.

In [217]:
#Test with high abundance genes and 1 batch
cellfeature="Condition"
featuredef="ALS"
numcells=10
genes=["MALAT1","PRNP"]
direction="KO"

pertres = perturb(ann_data, geneformer, cellfeature, featuredef, numcells, genes, direction)

2026-02-17 16:47:17,978 - INFO:helical.models.geneformer.model:Processing data for Geneformer.
2026-02-17 16:47:22,842 - INFO:helical.utils.mapping:Mapped 10847 / 10862 genes to Ensembl IDs.
2026-02-17 16:47:22,877 - INFO:helical.models.geneformer.geneformer_tokenizer:AnnData object with n_obs × n_vars = 10 × 10862
    obs: 'Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType', 'full_label', 'DGE_Group', 'Bakken_M1', 'data_merge_id', 'data_sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'Cellstates_LVL1', 'Cellstates_LVL2', 'Cellstates_LVL3', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'split'
    var: 'Biotype', 'Chromosome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ENSID', 'mt', 'n_cells', 'biotype', 'index', 'ensembl_id', 'ensembl_id_collapsed' has no column attribute 'filter_

[[-0.24247769  0.6253082   0.15658681  0.26449952  0.086252    0.2645227
   0.15443636  0.11593268  0.06215327]
 [-0.40467572  0.35478103 -0.05212227  0.0238226  -0.09875607  0.14718361
   0.27710855 -0.11666377  0.11366819]
 [-0.22974601  0.5836331  -0.29596362  0.15053438 -0.05747228  0.19160733
   0.29482406 -0.27274162  0.00995731]
 [-0.15189002  0.5300011  -0.15023896 -0.43523726  0.16206124  0.3420989
   0.6153344  -0.53381276 -0.00600063]
 [-0.2648031   0.699919   -0.49921492  0.22579224  0.01114558  0.16717437
   0.34864742 -0.23982584  0.02386237]
 [-0.32229376  0.5192339  -0.05769067  0.23877412  0.21429828  0.28201216
   0.1734842  -0.10895958  0.08197527]
 [-0.36139232 -0.25008833 -0.30000055 -0.4006857   0.02349068  0.16277032
   0.0588204   0.17809643  0.14813228]
 [-0.33855486  0.21490647 -0.36873412  0.19520333  0.08869382  0.25994748
   0.29300308 -0.68055505  0.10994177]
 [-0.17117804 -0.3412269   0.34036836 -0.16276564  0.08558202  0.4151724
   0.29655454 -0.04382376

In [220]:
print(pertres["base"][1:100,1])
print(pertres["modified"][1:100,1])

[-0.24247769 -0.40467572 -0.22974601 -0.15189002 -0.2648031  -0.32229376
 -0.36139232 -0.33855486 -0.17117804]
[-0.22542027 -0.40467572 -0.22077066 -0.14835392 -0.2627462  -0.3428965
 -0.35266832 -0.3436586  -0.16829352]
